# 03 - Test

This notebook executes a full speech roundtrip using the deployed Foundry-aligned AI Services account.

- Synthesize text to WAV
- Recognize speech from the generated WAV
- Save a JSON result in `outputs/`

**Pre-requisite:** run `02_configure.ipynb` first.

---

## Step 1 - Load environment variables

In [ ]:
import json
import os
from pathlib import Path

import azure.cognitiveservices.speech as speechsdk
from azure.identity import DefaultAzureCredential
from dotenv import load_dotenv

load_dotenv(dotenv_path="../env/.env")

if os.getenv("AZURE_AUTH_MODE", "").lower() != "aad":
    raise RuntimeError("This demo supports managed identity / AAD mode only.")

speech_endpoint = os.environ["AZURE_AI_ENDPOINT"]
voice_name = os.getenv("AZURE_SPEECH_VOICE", "en-US-AvaMultilingualNeural")

credential = DefaultAzureCredential()
token = credential.get_token("https://cognitiveservices.azure.com/.default").token

output_dir = Path("../outputs")
output_dir.mkdir(parents=True, exist_ok=True)

print("Environment loaded.")
print(f"Speech endpoint: {speech_endpoint}")
print(f"Voice: {voice_name}")

## Step 2 - Create Speech SDK configuration

The demo uses Azure AD token authentication (no account keys).

In [ ]:
speech_config = speechsdk.SpeechConfig(endpoint=speech_endpoint)
speech_config.authorization_token = token
speech_config.speech_synthesis_voice_name = voice_name
speech_config.speech_recognition_language = "en-US"

wav_path = output_dir / "speech01-roundtrip.wav"
json_path = output_dir / "speech01-roundtrip.json"

speech_config

## Step 3 - Synthesize text to audio

In [ ]:
input_text = "This demo validates Azure Speech from a Foundry aligned AI Services account."

audio_out = speechsdk.audio.AudioOutputConfig(filename=str(wav_path))
synthesizer = speechsdk.SpeechSynthesizer(speech_config=speech_config, audio_config=audio_out)
synth_result = synthesizer.speak_text_async(input_text).get()

if synth_result.reason != speechsdk.ResultReason.SynthesizingAudioCompleted:
    cancel = getattr(synth_result, "cancellation_details", None)
    cancel_reason = getattr(cancel, "reason", "unknown")
    error_details = getattr(cancel, "error_details", "")
    raise RuntimeError(
        f"Synthesis failed: reason={synth_result.reason}, cancellation={cancel_reason}, details={error_details}"
    )

print(f"Audio written to: {wav_path.resolve()}")

In [ ]:
import requests

# Optional REST alternative to Step 3 (keeps SDK method intact).
# This writes the same wav_path file, so run either SDK or REST synthesis before Step 4.
input_text = "This demo validates Azure Speech from a Foundry aligned AI Services account."

# Speech REST TTS route for this endpoint style.
tts_endpoint = speech_endpoint.rstrip("/") + "/tts/cognitiveservices/v1"
ssml = f"""<speak version='1.0' xml:lang='en-US'><voice name='{voice_name}'>{input_text}</voice></speak>"""

response = requests.post(
    tts_endpoint,
    headers={
        "Authorization": f"Bearer {token}",
        "Content-Type": "application/ssml+xml",
        "X-Microsoft-OutputFormat": "riff-24khz-16bit-mono-pcm",
        "User-Agent": "speech01-roundtrip-test",
    },
    data=ssml.encode("utf-8"),
    timeout=60,
)

if response.status_code < 200 or response.status_code >= 300:
    raise RuntimeError(
        f"Speech synthesis REST call failed with HTTP {response.status_code}.\n"
        f"TTS endpoint: {tts_endpoint}\n"
        f"Response: {response.text}"
    )

wav_path.write_bytes(response.content)
print(f"REST TTS endpoint used: {tts_endpoint}")
print(f"Audio written to: {wav_path.resolve()}")

## Step 4 - Recognize the synthesized audio and save result

In [ ]:
audio_in = speechsdk.audio.AudioConfig(filename=str(wav_path))
recognizer = speechsdk.SpeechRecognizer(speech_config=speech_config, audio_config=audio_in)
recognition = recognizer.recognize_once_async().get()

recognized_text = ""
if recognition.reason == speechsdk.ResultReason.RecognizedSpeech:
    recognized_text = recognition.text

result = {
    "input_text": input_text,
    "voice": voice_name,
    "wav_path": str(wav_path),
    "recognized_text": recognized_text,
    "recognition_reason": str(recognition.reason),
}

json_path.write_text(json.dumps(result, indent=2), encoding="utf-8")
print(f"Result JSON written to: {json_path.resolve()}")
result

In [ ]:
import requests

# Optional REST alternative to Step 4 (keeps SDK method intact).
# Uses the Foundry-aligned resource endpoint (same host as AZURE_AI_ENDPOINT).
stt_endpoint = (
    speech_endpoint.rstrip("/")
    + "/speechtotext/transcriptions:transcribe"
    + "?api-version=2024-11-15"
)

files = {
    "audio": (
        wav_path.name,
        wav_path.read_bytes(),
        "audio/wav",
    ),
    "definition": (
        None,
        json.dumps({"locales": ["en-US"]}),
        "application/json",
    ),
}

stt_response = requests.post(
    stt_endpoint,
    headers={
        "Authorization": f"Bearer {token}",
        "Accept": "application/json",
        "User-Agent": "speech01-roundtrip-test",
    },
    files=files,
    timeout=120,
)

if stt_response.status_code < 200 or stt_response.status_code >= 300:
    raise RuntimeError(
        f"Speech recognition REST call failed with HTTP {stt_response.status_code}.\n"
        f"STT endpoint: {stt_endpoint}\n"
        f"Response: {stt_response.text}"
    )

stt_payload = stt_response.json()
combined = stt_payload.get("combinedPhrases", [])
recognized_text = combined[0].get("text", "") if combined else ""

result = {
    "input_text": input_text,
    "voice": voice_name,
    "wav_path": str(wav_path),
    "recognized_text": recognized_text,
    "recognition_reason": "Success" if recognized_text else "NoMatch",
}

json_path.write_text(json.dumps(result, indent=2), encoding="utf-8")
print(f"REST STT endpoint used: {stt_endpoint}")
print(f"Result JSON written to: {json_path.resolve()}")
result

---

## ✅ Scenario complete

Outputs produced:

- `outputs/speech01-roundtrip.wav`
- `outputs/speech01-roundtrip.json`

Recommended follow-up: compare recognized text quality across multiple voices and locales.